In [337]:
import torch
from torchvision import datasets
import torchvision.transforms.v2 as v2
from torchvision.transforms.v2 import functional as F
from torchvision.transforms.v2 import ToPILImage
from PIL import Image
import os

In [185]:
one_image_dataset = datasets.ImageFolder(
    root="/Users/drmorsy/Downloads/Wadi Degla/test-pipeline/test",
    transform=None  # We'll apply transforms in splits
)

In [186]:
img = next(iter(one_image_dataset))[0]

In [187]:
def prep_image(img):
    """Convert image to proper format for transformations"""
    return v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)  # [0, 1] range
    ])(img)

## **1. Contrast adjustment**

In [90]:
def contrast_plus_30(img):
    img = prep_image(img)
    return F.adjust_contrast(img, contrast_factor=1.3)

# Apply your transformation
contrast_plus_30_img = v2.ToDtype(torch.uint8, scale=True)(contrast_plus_30(img))

# Convert to PIL Image
contrast_plus_30_img_pil = ToPILImage()(contrast_plus_30_img)
contrast_plus_30_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/contrast Pytorch (+30).png")   # PNG

In [91]:
def contrast_minus_85(img):
    img = prep_image(img)
    return F.adjust_contrast(img, contrast_factor=0.85)# Apply your transformation
contrast_minus_85_img = v2.ToDtype(torch.uint8, scale=True)((contrast_minus_85(img)))

# Convert to PIL Image
contrast_minus_85_img_pil = ToPILImage()(contrast_minus_85_img)
contrast_minus_85_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/contrast Pytorch (-85).png")   # PNG

## **2. Hue adjustment**

In [98]:
# Hue transformations (±180 → ±0.5)
def hue_plus_10(img):
    img = prep_image(img)
    # +10 in Photoshop = 10 / 360 = 0.0278
    return F.adjust_hue(img, hue_factor=0.0278)

In [97]:
hue_plus_10_img = v2.ToDtype(torch.uint8, scale=True)((hue_plus_10(img)))

# Convert to PIL Image
hue_plus_10_img_pil = ToPILImage()(hue_plus_10_img)
hue_plus_10_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/hue Pytorch (+10).png")   # PNG

In [104]:
def hue_minus_10(img):
    img = prep_image(img)
    # -10 in Photoshop = -10 / 360 = -0.0278
    return F.adjust_hue(img, hue_factor=-0.0278)

In [106]:
hue_minus_10_img = v2.ToDtype(torch.uint8, scale=True)((hue_minus_10(img)))

# Convert to PIL Image
hue_minus_10_img_pil = ToPILImage()(hue_minus_10_img)
hue_minus_10_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/hue Pytorch (-10).png")   # PNG

## **3. Saturation adjustment**

In [129]:
# Saturation transformations (±100)
def saturation_plus_30(img):
    img = prep_image(img)
    # +30 in Photoshop = 1.0 + (30/100) = 1.3
    return F.adjust_saturation(img, saturation_factor=1.3)

In [130]:
saturation_plus_30_img = v2.ToDtype(torch.uint8, scale=True)((saturation_plus_30(img)))

# Convert to PIL Image
saturation_plus_30_img_pil = ToPILImage()(saturation_plus_30_img)
saturation_plus_30_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/saturation Pytorch (+30).png")   # PNG

In [139]:
# Saturation transformations (±100)
def saturation_minus_30(img):
    img = prep_image(img)
    # -30 in Photoshop = 1.0 + (-30/100) = 0.7
    return F.adjust_saturation(img, saturation_factor=0.7)

In [141]:
saturation_minus_30_img = v2.ToDtype(torch.uint8, scale=True)((saturation_minus_30(img)))

# Convert to PIL Image
saturation_minus_30_img_pil = ToPILImage()(saturation_minus_30_img)
saturation_minus_30_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/saturation Pytorch (-30).png")   # PNG

## **4. Brightness**

In [169]:
def brightness_plus_30(img):
    img = prep_image(img)  # Assuming prep_image converts to Tensor
    # +50 in Photoshop (range ±150) = 1 + (50/150) = 1.3333
    return F.adjust_brightness(img, brightness_factor=1.3)

In [170]:
brightness_plus_30_img =  v2.ToDtype(torch.uint8, scale=True)((brightness_plus_30(img)))
# Convert to PIL Image
brightness_plus_30_img_pil = ToPILImage()(brightness_plus_30_img)
brightness_plus_30_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/brightness Pytorch (+30).png")   # PNG

In [180]:
def brightness_minus_30(img):
    img = prep_image(img)  # Assuming prep_image converts to Tensor
    # +50 in Photoshop (range ±150) = 1 + (50/150) = 1.3333
    return F.adjust_brightness(img, brightness_factor=.7)

In [181]:
brightness_minus_30_img =  v2.ToDtype(torch.uint8, scale=True)((brightness_minus_30(img)))
# Convert to PIL Image
brightness_minus_30_img_pil = ToPILImage()(brightness_minus_30_img)
brightness_minus_30_img_pil.save("/Users/drmorsy/Downloads/Wadi Degla/data augment pytorch/brightness Pytorch (-30).png")   # PNG

## **5. Augmentation Pipeline**

**Resize then Augment**
- **When working with plant images of varying sizes, the recommended order is: resize first, then augment.**


1. **Consistency:** Resizing first ensures all images share the same dimensions, leading to consistent transformation behavior during augmentation.

2. **Computational Efficiency:** Augmentation is faster on smaller, uniform images, while large images use more memory and slow training.
  
3. **Avoids Distortion Cascading:** Resizing first provides a clean base, preventing amplified artifacts (blur, aliasing, edge blanks) that occur when resizing after augmentation.
  

In [331]:
from torch.utils.data import DataLoader
from torchvision import transforms
aug_one_image_dataset = datasets.ImageFolder(
    root="/Users/drmorsy/Downloads/Wadi Degla/test-pipeline/test",
)

In [332]:
# Optimized Plant Identification Augmentation Pipeline for Training
augmentation_pipeline = v2.Compose([
    # -----------------------------------------------------------
    # Stage 1: Input Preparation & Spatial Augmentations
    # -----------------------------------------------------------
    v2.ToImage(),  # Convert PIL/HWC to torch.Tensor(CHW, uint8)
    
    # Core spatial augmentations - applied in uint8 for speed/quality
    v2.RandomResizedCrop(
        size=(224, 224), 
        scale=(0.2, 1.0),
        ratio=(0.75, 1.33),
        antialias=True
    ),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomRotation(
        degrees=(-90, 90), 
        interpolation=v2.InterpolationMode.BILINEAR
    ),
    
    # -----------------------------------------------------------
    # Stage 2: Color Operations
    # -----------------------------------------------------------
    v2.ToDtype(torch.float32, scale=True),  # Convert to [0,1] float32
    
    # Color augmentations with plant-specific adjustments
    v2.ColorJitter(
        brightness=(0.7, 1.3),
        contrast=(0.85, 1.3),
        saturation=(0.7, 1.3),
        hue=(-0.0278, 0.0278)
    ),
    
    # -----------------------------------------------------------
    # Stage 3: Normalization (CRUCIAL for training)
    # -----------------------------------------------------------
    # v2.Normalize(
    #     mean=[0.485, 0.456, 0.406], 
    #     std=[0.229, 0.224, 0.225]
    # )
])

In [333]:
aug_one_image_dataset.transform = augmentation_pipeline

In [334]:
# Directory to save augmented images
save_dir = "/Users/drmorsy/Downloads/Wadi Degla/Augmented images 500*500"  # Change this to your desired path

In [335]:
# Create a DataLoader to iterate through images
dataloader = DataLoader(aug_one_image_dataset, batch_size=20, shuffle=False)

In [336]:
# Process and save images
c = 0 
for images, labels in dataloader:
    # Convert tensor back to PIL Image for saving
    for i in range(len(images)):
        to_pil = transforms.ToPILImage()
        pil_image = to_pil(images[i])  
        # Save the image
        save_path = os.path.join(save_dir, f"augmented_image_{c:04d}.png")
        pil_image.save(save_path)
        c+=1
        if i >= 49:  # Save first 50 images as example
            break
    
print("Finished saving augmented images!")   

Finished saving augmented images!
